# EDA — ASL Fingerspelling Dataset
CSS 324 Final Project

Exploratory analysis of the landmark dataset extracted from the Kaggle ASL Alphabet dataset using MediaPipe. 19,592 samples across 26 classes (A-Y, no J/Z, plus SPACE and DELETE).

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE

df = pd.read_csv('/content/drive/MyDrive/asl_project/data/landmarks_dataset1.csv')
df = df[df['letter'] != 'NOTHING']

print(f'{len(df)} samples, {df["letter"].nunique()} classes')
print(df['letter'].value_counts().sort_index())

## Class Distribution

Each class had up to 1000 images attempted. Not all produced valid landmarks — MediaPipe fails on blurry, poorly lit, or oddly angled hands.

In [ ]:
counts = df['letter'].value_counts().sort_index()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# samples per class
axes[0].bar(counts.index, counts.values, color='steelblue')
axes[0].axhline(counts.mean(), color='red', linestyle='--', label=f'mean ({counts.mean():.0f})')
axes[0].set_xlabel('Letter')
axes[0].set_ylabel('Samples extracted')
axes[0].set_title('Samples per class')
axes[0].legend()

# dropout per class (1000 attempted per class)
dropout_pct = ((1000 - counts) / 1000 * 100).sort_values(ascending=False)
axes[1].bar(dropout_pct.index, dropout_pct.values, color='salmon')
axes[1].axhline(dropout_pct.mean(), color='darkred', linestyle='--', label=f'mean ({dropout_pct.mean():.1f}%)')
axes[1].set_xlabel('Letter')
axes[1].set_ylabel('Dropout %')
axes[1].set_title('MediaPipe detection failure rate per class')
axes[1].legend()

plt.tight_layout()
plt.savefig('/content/drive/MyDrive/asl_project/class_distribution.png', dpi=150)
plt.show()

print(f'Min samples: {counts.min()} ({counts.idxmin()})')
print(f'Max samples: {counts.max()} ({counts.idxmax()})')
print(f'\nHighest dropout:')
print(dropout_pct.head(5).to_string())

## Average Hand Pose Per Class

For each class, we compute the mean of all 21 landmark coordinates and draw the hand skeleton. This shows what the model is actually learning — not pixels, but the geometric shape of each sign.

In [ ]:
# MediaPipe hand skeleton connections
CONNECTIONS = [
    (0,1),(1,2),(2,3),(3,4),
    (0,5),(5,6),(6,7),(7,8),
    (0,9),(9,10),(10,11),(11,12),
    (0,13),(13,14),(14,15),(15,16),
    (0,17),(17,18),(18,19),(19,20),
    (5,9),(9,13),(13,17)
]

def plot_hand(ax, xs, ys, title=''):
    for a, b in CONNECTIONS:
        ax.plot([xs[a], xs[b]], [ys[a], ys[b]], 'b-', lw=1.5, alpha=0.6)
    ax.scatter(xs, ys, s=20, c='navy', zorder=5)
    ax.set_title(title, fontsize=10, fontweight='bold')
    ax.set_aspect('equal')
    ax.axis('off')

letters = sorted(df['letter'].unique())
cols = 6
rows = (len(letters) + cols - 1) // cols

fig, axes = plt.subplots(rows, cols, figsize=(cols * 2.5, rows * 2.8))
axes = axes.flatten()

for i, letter in enumerate(letters):
    sub = df[df['letter'] == letter]
    xs = [sub[f'x{j}'].mean() for j in range(21)]
    ys = [-sub[f'y{j}'].mean() for j in range(21)]  # flip y for natural orientation
    plot_hand(axes[i], xs, ys, title=letter)

for j in range(len(letters), len(axes)):
    axes[j].axis('off')

plt.suptitle('Mean landmark pose per class (wrist at origin)', fontsize=13, y=1.01)
plt.tight_layout()
plt.savefig('/content/drive/MyDrive/asl_project/avg_poses.png', dpi=150, bbox_inches='tight')
plt.show()

## Visually Similar Classes

M, N, S, and T are the worst-performing classes in our model (F1 scores 91-94%). Plotting their mean poses shows why — the landmark geometry is nearly identical. These signs differ mainly in which fingers are tucked under, which produces similar x/y coordinates but different z (depth), and depth is noisy from a standard webcam.

In [ ]:
focus = ['M', 'N', 'S', 'T']

fig, axes = plt.subplots(1, len(focus), figsize=(12, 3))

for ax, letter in zip(axes, focus):
    sub = df[df['letter'] == letter]
    xs = [sub[f'x{j}'].mean() for j in range(21)]
    ys = [-sub[f'y{j}'].mean() for j in range(21)]
    plot_hand(ax, xs, ys, title=f'{letter}  (n={len(sub)})')

plt.suptitle('Hardest letters — visually similar mean poses', fontsize=11)
plt.tight_layout()
plt.savefig('/content/drive/MyDrive/asl_project/confused_letters.png', dpi=150)
plt.show()

# pairwise distance between class means
class_means = {}
for letter in letters:
    sub = df[df['letter'] == letter]
    feat_cols = [f'x{i}' for i in range(21)] + [f'y{i}' for i in range(21)] + [f'z{i}' for i in range(21)]
    class_means[letter] = sub[feat_cols].mean().values

# find closest pairs
pairs = []
for i, l1 in enumerate(letters):
    for l2 in letters[i+1:]:
        d = np.linalg.norm(class_means[l1] - class_means[l2])
        pairs.append((d, l1, l2))

pairs.sort()
print('Most similar class pairs (lowest mean landmark distance):')
for d, l1, l2 in pairs[:8]:
    print(f'  {l1} vs {l2}: {d:.4f}')

## PCA — Feature Space Structure

Principal Component Analysis reduces the 63 landmark features to 2 dimensions. Well-separated clusters mean the features are discriminative — the model should be able to learn clear decision boundaries.

In [ ]:
feat_cols = [f'x{i}' for i in range(21)] + [f'y{i}' for i in range(21)] + [f'z{i}' for i in range(21)]
X = df[feat_cols].values
labels = df['letter'].values

pca = PCA(n_components=2)
X_pca = pca.fit_transform(X)

fig, ax = plt.subplots(figsize=(12, 8))
for letter in sorted(set(labels)):
    mask = labels == letter
    ax.scatter(X_pca[mask, 0], X_pca[mask, 1], alpha=0.25, s=6, label=letter)

ax.set_xlabel(f'PC1 ({pca.explained_variance_ratio_[0]*100:.1f}% variance)')
ax.set_ylabel(f'PC2 ({pca.explained_variance_ratio_[1]*100:.1f}% variance)')
ax.set_title('PCA of 63 landmark features')
ax.legend(bbox_to_anchor=(1.02, 1), loc='upper left', fontsize=7, ncol=2, markerscale=3)
plt.tight_layout()
plt.savefig('/content/drive/MyDrive/asl_project/pca.png', dpi=150, bbox_inches='tight')
plt.show()

total_var = sum(pca.explained_variance_ratio_[:2]) * 100
print(f'Variance explained by 2 PCs: {total_var:.1f}%')

# how many PCs to get 90% variance?
pca_full = PCA().fit(X)
cumvar = np.cumsum(pca_full.explained_variance_ratio_)
n90 = np.argmax(cumvar >= 0.90) + 1
print(f'PCs needed for 90% variance: {n90}')

## t-SNE — Non-linear Embedding

t-SNE reveals cluster structure that PCA misses. Tight, well-separated clusters indicate the landmark features cleanly distinguish between classes. Overlapping clusters (like M/N/S/T) explain where the model makes mistakes.

In [ ]:
# subsample to keep it fast
np.random.seed(42)
idx = np.random.choice(len(X), 4000, replace=False)
X_s = X[idx]
labels_s = labels[idx]

print('Running t-SNE on 4000 samples...')
tsne = TSNE(n_components=2, random_state=42, perplexity=40, n_iter=1000)
X_tsne = tsne.fit_transform(X_s)

fig, ax = plt.subplots(figsize=(12, 8))
for letter in sorted(set(labels_s)):
    mask = labels_s == letter
    ax.scatter(X_tsne[mask, 0], X_tsne[mask, 1], alpha=0.4, s=8, label=letter)

ax.set_title('t-SNE of hand landmarks (4000 samples)')
ax.legend(bbox_to_anchor=(1.02, 1), loc='upper left', fontsize=7, ncol=2, markerscale=3)
ax.axis('off')
plt.tight_layout()
plt.savefig('/content/drive/MyDrive/asl_project/tsne.png', dpi=150, bbox_inches='tight')
plt.show()

## Feature Statistics

Which landmarks have the most variance across classes? High-variance landmarks are the most informative features for classification.

In [ ]:
# variance of each landmark coordinate across the full dataset
feat_var = df[feat_cols].var().values

# group by landmark index (each landmark has x, y, z)
landmark_var = []
for i in range(21):
    v = feat_var[i] + feat_var[i+21] + feat_var[i+42]  # x + y + z variance
    landmark_var.append(v)

plt.figure(figsize=(10, 4))
plt.bar(range(21), landmark_var, color='mediumseagreen')
plt.xlabel('Landmark index')
plt.ylabel('Total variance (x+y+z)')
plt.title('Which landmarks vary most across the dataset')
plt.xticks(range(21))
plt.tight_layout()
plt.savefig('/content/drive/MyDrive/asl_project/landmark_variance.png', dpi=150)
plt.show()

most_var = np.argsort(landmark_var)[::-1][:5]
print('Most informative landmarks:')
lm_names = [
    'Wrist', 'Thumb CMC', 'Thumb MCP', 'Thumb IP', 'Thumb tip',
    'Index MCP', 'Index PIP', 'Index DIP', 'Index tip',
    'Middle MCP', 'Middle PIP', 'Middle DIP', 'Middle tip',
    'Ring MCP', 'Ring PIP', 'Ring DIP', 'Ring tip',
    'Pinky MCP', 'Pinky PIP', 'Pinky DIP', 'Pinky tip'
]
for i in most_var:
    print(f'  Landmark {i} ({lm_names[i]}): variance = {landmark_var[i]:.4f}')

## Related Work

**[1] Zhang et al. (2020) — MediaPipe Hands: On-device Real-time Hand Tracking**
*Google Research / CVPR 2020 workshop*

Describes the pipeline we use for feature extraction. MediaPipe Hands runs a two-stage detector: a palm detection CNN followed by a hand landmark model that regresses 21 3D keypoints. Key design choice: the landmark model takes a tight crop around the detected hand, which makes it robust to scale and position variation. This is why wrist-relative normalization works well — the model already handles gross position variation, and our normalization removes the remaining scale differences between individuals.

---

**[2] Taskiran et al. (2018) — Hand Gesture Recognition Using Computer Vision**
*Journal of Electrical Engineering and Computer Science*

Compares CNN-based image classification against landmark-based approaches with SVM and Random Forest on gesture recognition tasks. Finds that landmark + classical ML reaches similar accuracy to CNNs on datasets with fewer than ~50k samples, while being 10-20x faster at inference. This supports our decision to use SVM on 63 landmark features rather than training a CNN from scratch — we get ~98% accuracy with a model that runs in under 1ms per frame, which is critical for real-time feedback.

---

**[3] Bantupalli & Xie (2018) — American Sign Language Recognition Using Deep Learning**
*IEEE International Conference on Big Data*

One of the earlier papers to apply deep CNNs directly to ASL alphabet recognition from RGB images. Achieves ~98% test accuracy on the same Kaggle ASL Alphabet dataset we use, but requires full image classification (224×224 pixels through ResNet-style architecture) rather than 63 landmark coordinates. Their preprocessing normalizes hand region crops but still operates on raw pixels. Our approach achieves the same accuracy with dramatically lower compute — relevant because the final app needs to run in real-time on a webcam stream.